# Train a Dataset Builder zip

Trains yolov8n-seg on this dataset and gives back `best.onnx`.

1. Set **Runtime → Change runtime type → GPU**.
2. In the **Dataset zip** cell paste the Google Drive link of **your dataset zip** (shared as *Anyone with the link*): Colab downloads it far faster than an upload. Left empty, the cell asks you to upload the zip instead.
3. Run the cells in order, or **Runtime → Run all**.

In [ ]:
!pip install -q ultralytics

In [ ]:
#@title Dataset zip
#@markdown A Google Drive link to the zip, shared as Anyone with the link. Left empty, the zip is uploaded from this computer.
drive_link = ""  #@param {type:"string"}
from google.colab import files
if drive_link.strip():
    import gdown
    zip_name = gdown.download(drive_link.strip(), "/content/dataset.zip", fuzzy=True)
    assert zip_name, "Drive did not give the zip: share it as Anyone with the link, or leave the link empty to upload it."
else:
    up = files.upload()  # pick your dataset zip
    zip_name = next(iter(up))
!rm -rf /content/ds && mkdir -p /content/ds && unzip -q "{zip_name}" -d /content/ds
!cat /content/ds/data.yaml

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n-seg.pt")
model.train(data="/content/ds/data.yaml", imgsz=512, epochs=200, patience=20, batch=32, fliplr=0.0, mosaic=1.0, project="/content/runs", name="seg", exist_ok=True)

In [ ]:
# Fine-tune from the best weights at a lower learning rate. optimizer="auto" would ignore lr0, so it is named.
tuned = YOLO("/content/runs/seg/weights/best.pt")
tuned.train(data="/content/ds/data.yaml", imgsz=512, epochs=40, patience=10, batch=32, optimizer="AdamW", lr0=0.0002, lrf=0.1, cos_lr=True, warmup_epochs=1, fliplr=0.0, mosaic=1.0, project="/content/runs", name="seg-tune", exist_ok=True)

In [ ]:
# The fine-tune is kept only if it scores better on the valid split: it does not always help.
first = YOLO("/content/runs/seg/weights/best.pt").val(data="/content/ds/data.yaml", imgsz=512).fitness
second = YOLO("/content/runs/seg-tune/weights/best.pt").val(data="/content/ds/data.yaml", imgsz=512).fitness
run = "/content/runs/seg-tune/weights/" if second > first else "/content/runs/seg/weights/"
print("fine-tune", "kept" if second > first else "dropped", round(first, 4), "->", round(second, 4))
best = YOLO(run + "best.pt")
best.val(data="/content/ds/data.yaml", split="test", imgsz=512)
best.export(format="onnx", imgsz=512, opset=12, simplify=True)
files.download(run + "best.onnx")
files.download(run + "best.pt")

In [ ]:
#@title Save to Google Drive
#@markdown Copies best.pt and best.onnx to My Drive / Dataset Builder, so they outlive this session. Colab asks for access to Drive.
save_to_drive = True  #@param {type:"boolean"}
if save_to_drive:
    import os, shutil, time
    from google.colab import drive
    drive.mount("/content/drive")
    kept = "/content/drive/MyDrive/Dataset Builder/" + time.strftime("seg-%Y%m%d-%H%M")
    os.makedirs(kept, exist_ok=True)
    for weights in ("best.pt", "best.onnx"):
        shutil.copy(run + weights, kept)
    print("saved to", kept)

The ONNX keeps no class names: they are `names` in `data.yaml`, in that order.

A run that stopped resumes with `YOLO("/content/runs/seg/weights/last.pt").train(resume=True)`, or `/content/runs/seg-tune/weights/last.pt` for the fine-tune.

`best.pt` trains on later, with more epochs or new images: upload it, then `YOLO("best.pt").train(data=..., epochs=...)`.